# RescueBite AI Model A — Data Audit Notebook

> **Synthetic Data Disclaimer**: This dataset is synthetically generated for demonstration and simulation purposes. All metrics and evaluations are synthetic-data experiments and do not represent live operational guarantees.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Robust path resolution from notebook directory
current_dir = Path.cwd()
repo_root = current_dir if (current_dir / 'dataset').exists() else current_dir.parent.parent

demand_path = repo_root / 'dataset' / 'ngo_demand_daily.csv'
ngos_path = repo_root / 'dataset' / 'ngos.csv'
holidays_path = repo_root / 'dataset' / 'holidays.csv'

print('Resolved paths:')
print('Demand:', demand_path.resolve())
print('NGOs:', ngos_path.resolve())
print('Holidays:', holidays_path.resolve())

df_demand = pd.read_csv(demand_path)
df_ngos = pd.read_csv(ngos_path)
df_holidays = pd.read_csv(holidays_path)
df_demand['date'] = pd.to_datetime(df_demand['date'])

## 1. Shapes, Columns & Dtypes

In [ ]:
print('--- SHAPES ---')
print(f'Demand: {df_demand.shape}')
print(f'NGOs: {df_ngos.shape}')
print(f'Holidays: {df_holidays.shape}')

print('\n--- COLUMNS & DTYPES ---')
print(df_demand.dtypes)

## 2. Null Values & Duplicates Check

In [ ]:
print('Demand null count:')
print(df_demand.isnull().sum())

print('\nDuplicate rows:', df_demand.duplicated().sum())
print('Duplicate (ngo_id, date) composite keys:', df_demand.duplicated(subset=['ngo_id', 'date']).sum())

## 3. Target Statistics (`meals_requested`)

In [ ]:
target = df_demand['meals_requested']
print('Min:', target.min())
print('Max:', target.max())
print('Mean:', round(target.mean(), 2))
print('Median:', target.median())
print('Std Dev:', round(target.std(), 2))
print('Negative/Invalid (<0):', (target < 0).sum())

## 4. Date Range & Grid Completeness (404 Missing Rows Policy)

In [ ]:
min_d, max_d = df_demand['date'].min(), df_demand['date'].max()
days_span = (max_d - min_d).days + 1
unique_ngos = df_demand['ngo_id'].nunique()
expected_grid = unique_ngos * days_span
actual_rows = len(df_demand)
missing_rows = expected_grid - actual_rows

print(f'Date range: {min_d.date()} to {max_d.date()} ({days_span} days)')
print(f'Unique NGOs: {unique_ngos}')
print(f'Expected grid rows: {expected_grid}')
print(f'Actual rows: {actual_rows}')
print(f'Missing NGO-date records: {missing_rows}')
print('CRITICAL POLICY: Missing rows are NOT automatically zero-demand days.')

## 5. Relational Join Integrity

In [ ]:
demand_ngos = set(df_demand['ngo_id'].unique())
master_ngos = set(df_ngos['ngo_id'].unique())
print('Demand NGOs missing from ngos.csv:', demand_ngos - master_ngos)
print('Master NGOs with no demand records:', master_ngos - demand_ngos)

## 6. Daily Demand Trend Visualization

In [ ]:
daily_agg = df_demand.groupby('date')['meals_requested'].sum()
plt.figure(figsize=(12, 4))
plt.plot(daily_agg.index, daily_agg.values, color='#0F766E', linewidth=1.5)
plt.title('Total Daily NGO Meal Demand Over Time (540 Days)')
plt.xlabel('Date')
plt.ylabel('Meals Requested')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()